# Phase 3, experiment 11 (part 2): blending the GNN with LightGBM

Run `13_gnn.ipynb` first. It saves the GNN's predictions to `../outputs/`. This notebook trains the
default LightGBM on the same folds, rank-averages the two models with different weights, and scores
the result. (The two are in separate notebooks because training LightGBM and PyTorch in the same
session crashes the kernel on this Mac.)

`blend = (1 - w) x LightGBM rank + w x GNN rank`, where `w` is the GNN's weight. `w = 0` is LightGBM
alone and `w = 1` is the GNN alone. As before, an option is worth keeping only if it raises the fold
mean without lowering fold 2.

## 1. Load

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

SEED = 42
DATA_DIR = "../data"
OUT_DIR = "../outputs"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
raw_cols = [c for c in train.columns if c.startswith("feat_")]

labeled = train[train["label"].notna()].copy()
labeled["label"] = labeled["label"].astype(int)

gnn_fold = pd.read_csv(f"{OUT_DIR}/gnn_fold_predictions.csv")
gnn_test = pd.read_csv(f"{OUT_DIR}/gnn_test_predictions.csv")

## 2. Blend on the folds

In [2]:
FOLDS = [
    {"name": "fold 1", "train_last": 14, "valid_last": 21},
    {"name": "fold 2", "train_last": 21, "valid_last": 28},
    {"name": "fold 3", "train_last": 28, "valid_last": 35},
]
fold_names = [f["name"] for f in FOLDS]
GNN_WEIGHTS = [0.0, 0.1, 0.2, 0.35, 0.5, 1.0]

def to_rank(scores):
    # Position of each score (1 = lowest) divided by the count, so it runs from 0 to 1
    return pd.Series(scores).rank().values / len(scores)

score_rows = []
for fold in FOLDS:
    tr = labeled[labeled["time_step"] <= fold["train_last"]]
    va = labeled[(labeled["time_step"] > fold["train_last"]) & (labeled["time_step"] <= fold["valid_last"])]

    model = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
    model.fit(tr[raw_cols], tr["label"])
    lgbm_rank = to_rank(model.predict_proba(va[raw_cols])[:, 1])

    # GNN scores for the same rows, in the same order
    this_fold = gnn_fold[gnn_fold["fold"] == fold["name"]].set_index("txId")["gnn_score"]
    gnn_rank = to_rank(this_fold.loc[va["txId"]].values)

    in_24 = (va["time_step"] == 24).values
    for w in GNN_WEIGHTS:
        blend = (1 - w) * lgbm_rank + w * gnn_rank
        row = {"gnn_weight": w, "fold": fold["name"], "val_auc": roc_auc_score(va["label"], blend)}
        if fold["name"] == "fold 2":
            row["auc_step_24"] = roc_auc_score(va["label"][in_24], blend[in_24])
        score_rows.append(row)

scores = pd.DataFrame(score_rows)
results = scores.pivot(index="gnn_weight", columns="fold", values="val_auc")
results["mean"] = results[fold_names].mean(axis=1)
results["change_mean"] = results["mean"] - results.loc[0.0, "mean"]
results["change_fold2"] = results["fold 2"] - results.loc[0.0, "fold 2"]
results["auc_step_24"] = scores.dropna(subset=["auc_step_24"]).set_index("gnn_weight")["auc_step_24"]
results.round(4)

fold,fold 1,fold 2,fold 3,mean,change_mean,change_fold2,auc_step_24
gnn_weight,,,,,,,
0.00,0.9888,0.9572,0.9965,0.9808,0.0000,0.0000,0.7866
0.10,0.9901,0.9544,0.9978,0.9808,-0.0001,-0.0028,0.7796
0.20,0.9906,0.9508,0.9982,0.9799,-0.0010,-0.0064,0.7720
0.35,0.9897,0.9439,0.9980,0.9772,-0.0036,-0.0133,0.7501
0.50,0.9867,0.9364,0.9972,0.9734,-0.0074,-0.0208,0.7277
1.00,0.9562,0.9065,0.9807,0.9478,-0.0330,-0.0507,0.6530


## 3. Submission

This retrains LightGBM on steps 1 to 35 and blends its test predictions with the GNN's at the weight
below. 

In [7]:
CHOSEN_GNN_WEIGHT = 0.1

final_model = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
final_model.fit(labeled[raw_cols], labeled["label"])
lgbm_test_rank = to_rank(final_model.predict_proba(test[raw_cols])[:, 1])
gnn_test_rank = to_rank(gnn_test.set_index("txId")["gnn_score"].loc[test["txId"]].values)

test_pred = (1 - CHOSEN_GNN_WEIGHT) * lgbm_test_rank + CHOSEN_GNN_WEIGHT * gnn_test_rank
submission = pd.DataFrame({"index": test["index"], "target": test_pred})

assert len(submission) == 15329, "wrong number of rows"
assert list(submission.columns) == ["index", "target"], "wrong columns"
assert submission["target"].notna().all(), "NaN in predictions"
assert (submission["index"].values == sample_sub["index"].values).all(), "index does not match sample submission"

SUB_PATH = "../submissions/sub06_gnn_blend.csv"
submission.to_csv(SUB_PATH, index=False)
print("Saved", SUB_PATH, "with GNN weight", CHOSEN_GNN_WEIGHT)

Saved ../submissions/sub06_gnn_blend.csv with GNN weight 0.1


## Findings

- **Does any blend raise the fold mean without lowering fold 2?** No. A GNN weight of 0.1 leaves the mean
  unchanged (0.9808, -0.0001) and lowers fold 2 by 0.0028. Higher weights lower both: at 0.2 the mean
  falls by 0.0010 and fold 2 by 0.0064, and at 0.5 by 0.0074 and 0.0208. Small weights do raise folds 1
  and 3 slightly (at 0.1: +0.0013 and +0.0013; at 0.2: +0.0018 and +0.0017), the same trade-off seen with
  every other change.
- **What happens on step 24?** It falls as the GNN weight rises: 0.787 for LightGBM alone, 0.780 at 0.1,
  0.772 at 0.2 and 0.653 for the GNN alone.
- **Public leaderboard AUC for `sub06_gnn_blend.csv` (GNN weight 0.1):** 0.95037. This is 0.0038 above
  LightGBM alone (`sub01_baseline.csv`, 0.94660) and 0.0027 above the previous best (`sub02_degree.csv`,
  0.94770), the best score so far. It is the first time the leaderboard and the fold mean disagree since
  the folds were introduced: the folds called this blend level, the leaderboard shows a gain. A gain of
  this size is at the edge of what could be noise on 30% of the test set, so it is suggestive but not
  conclusive. One possible reading is that the test period behaves more like folds 1 and 3, where the
  blend helps slightly.

